In [2]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/test-d/cleaned_data_test.csv
/kaggle/input/item-data-2/cleaned_data.csv


In [3]:
pip install -U sentence-transformers transformers huggingface_hub tokenizers

Note: you may need to restart the kernel to use updated packages.


In [4]:
import pandas as pd

In [5]:
data=pd.read_csv("/kaggle/input/item-data-2/cleaned_data.csv")

In [6]:
data.head()

,sample_id,value,item_data,log_price
0,33127,72.0,"La Victoria Green Taco Sauce Mild, 12 Ounce (P...",1.587192
1,198967,32.0,"Salerno Cookies, The Original Butter Cookies, ...",2.574138
2,261251,11.4,"Bear Creek Hearty Soup Bowl, Creamy Chicken wi...",0.678034
3,55858,11.25,Judee’s Blue Cheese Powder 11.25 oz - Gluten-F...,3.412467
4,292686,12.0,"kedem Sherry Cooking Wine, 12.7 Ounce - 12 per...",4.197052


In [7]:
from sklearn.model_selection import train_test_split
X = data[['item_data', 'value']]
y = data['log_price']

In [8]:
X['value'] = pd.to_numeric(X['value'], errors='coerce') 

/tmp/ipykernel_101/3801193570.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['value'] = pd.to_numeric(X['value'], errors='coerce')


In [9]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X['value_standardized'] = scaler.fit_transform(X[['value']])

/tmp/ipykernel_101/2416140022.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['value_standardized'] = scaler.fit_transform(X[['value']])


In [10]:
X.drop("value",axis=1,inplace=True)

/tmp/ipykernel_101/3947267770.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X.drop("value",axis=1,inplace=True)


In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [12]:
X_train

,item_data,value_standardized
61160,INSHELL Texas Fresh Pecans 3-Pound Burlap Bag ...,-0.016739
3067,"Bear Creek Soup Mix, Cheddar Broccoli, 11.2 Ou...",-0.090894
43932,"Parsley Leaf Tea (Loose) (4 oz, ZIN: 511922) -...",-0.089281
25303,"De Cecco Semolina Pasta, Linguine No.7, 5 Poun...",0.531362
31741,"Earth Exotics, Ams French Beans, 8 Ounce",-0.097342
...,...,...
62570,Take 5 Snack Size Candy Bars - 11.25oz - PACK ...,-0.022784
38158,Blue Dragon Thai Chilli Dipping Sauce 190g,-0.099961
860,Kraft Thousand Island Salad Dressing Family Si...,-0.065101
15795,Junior Caramels 3.6oz Theater Box: 12 Count Ke...,-0.089281


In [13]:
import torch
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import numpy as np
import os
os.environ['HF_HUB_DISABLE_TELEMETRY'] = '1'
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained(
    'google-bert/bert-base-uncased',
    use_fast=True,
    local_files_only=False
)
bert_model = AutoModel.from_pretrained("google-bert/bert-base-uncased")
bert_model.eval()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)
bert_model.to(device)

def get_bert_embeddings(texts, tokenizer, bert_model, batch_size=32, max_length=128):
    embeddings = []
    for i in tqdm(range(0, len(texts), batch_size)):
        batch = texts[i:i+batch_size]
        encoded = tokenizer(
            batch,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt'
        )
        input_ids = encoded['input_ids'].to(device)
        attention_mask = encoded['attention_mask'].to(device)
        with torch.no_grad():
            outputs = bert_model(input_ids=input_ids, attention_mask=attention_mask)
            # Use [CLS] token embedding as sentence embedding
            cls_embeddings = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(cls_embeddings)
    return np.vstack(embeddings)


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

2025-10-10 23:42:35.936854: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1760139756.124125     101 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1760139756.176214     101 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

cuda


In [14]:
X_train_emb = get_bert_embeddings(X_train['item_data'].tolist(), tokenizer, bert_model)
X_test_emb = get_bert_embeddings(X_test['item_data'].tolist(), tokenizer, bert_model)

100%|██████████| 406/406 [00:46<00:00,  8.68it/s]


In [15]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

# -----------------------------
# 1. Convert embeddings to tensors
# -----------------------------
X_train_tensor = torch.tensor(X_train_emb, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)  # shape (n_samples, 1)

X_test_tensor = torch.tensor(X_test_emb, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# -----------------------------
# 2. Define the MLP
# -----------------------------
class MLPRegressor(nn.Module):
    def __init__(self, input_dim):
        super(MLPRegressor, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1)  # regression output
        )

    def forward(self, x):
        return self.model(x)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
t_model = MLPRegressor(input_dim=X_train_emb.shape[1]).to(device)

# -----------------------------
# 3. Loss and optimizer
# -----------------------------
criterion = nn.MSELoss()
optimizer = optim.Adam(t_model.parameters(), lr=1e-3)

# -----------------------------
# 4. Training loop
# -----------------------------
epochs = 50
for epoch in range(epochs):
    t_model.train()
    running_loss = 0
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = t_model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * X_batch.size(0)

    epoch_loss = running_loss / len(train_loader.dataset)
    print(f"Epoch {epoch+1}/{epochs}, Loss: {epoch_loss:.4f}")

# -----------------------------
# 5. Evaluation
# -----------------------------
t_model.eval()
with torch.no_grad():
    y_pred = t_model(X_test_tensor.to(device)).cpu().numpy()
    y_true = y_test_tensor.numpy()

def smape(y_true, y_pred):
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2
    diff = np.abs(y_true - y_pred) / denominator
    diff[denominator == 0] = 0
    return np.mean(diff) * 100

import numpy as np
mae = np.mean(np.abs(y_true - y_pred))
rmse = np.sqrt(np.mean((y_true - y_pred)**2))
r2 = 1 - np.sum((y_true - y_pred)**2)/np.sum((y_true - np.mean(y_true))**2)
smape_val = smape(y_true, y_pred)

print(f"MAE: {mae:.4f}, RMSE: {rmse:.4f}, R2: {r2:.4f}, SMAPE: {smape_val:.2f}%")


Epoch 1/50, Loss: 0.9692
Epoch 2/50, Loss: 0.8784
Epoch 3/50, Loss: 0.8439
Epoch 4/50, Loss: 0.8265
Epoch 5/50, Loss: 0.8114
Epoch 6/50, Loss: 0.7979
Epoch 7/50, Loss: 0.7842
Epoch 8/50, Loss: 0.7679
Epoch 9/50, Loss: 0.7629
Epoch 10/50, Loss: 0.7524
Epoch 11/50, Loss: 0.7387
Epoch 12/50, Loss: 0.7293
Epoch 13/50, Loss: 0.7177
Epoch 14/50, Loss: 0.7135
Epoch 15/50, Loss: 0.7027
Epoch 16/50, Loss: 0.6949
Epoch 17/50, Loss: 0.6809
Epoch 18/50, Loss: 0.6765
Epoch 19/50, Loss: 0.6692
Epoch 20/50, Loss: 0.6554
Epoch 21/50, Loss: 0.6497
Epoch 22/50, Loss: 0.6442
Epoch 23/50, Loss: 0.6318
Epoch 24/50, Loss: 0.6295
Epoch 25/50, Loss: 0.6198
Epoch 26/50, Loss: 0.6095
Epoch 27/50, Loss: 0.6035
Epoch 28/50, Loss: 0.5976
Epoch 29/50, Loss: 0.5901
Epoch 30/50, Loss: 0.5773
Epoch 31/50, Loss: 0.5746
Epoch 32/50, Loss: 0.5664
Epoch 33/50, Loss: 0.5574
Epoch 34/50, Loss: 0.5558
Epoch 35/50, Loss: 0.5471
Epoch 36/50, Loss: 0.5408
Epoch 37/50, Loss: 0.5324
Epoch 38/50, Loss: 0.5272
Epoch 39/50, Loss: 0.

In [16]:
test_data = pd.read_csv("/kaggle/input/test-d/cleaned_data_test.csv")

In [17]:
test_data.head()

,sample_id,value,item_data
0,100179,10.5,Rani 14-Spice Eshamaya's Mango Chutney (Indian...
1,245611,2.0,Natural MILK TEA Flavoring extract by HALO PAN...
2,146263,32.0,Honey Filled Hard Candy - Bulk Pack 2 Pounds -...
3,95658,2.0,Vlasic Snack'mm's Kosher Dill 16 Oz (Pack of 2)
4,36806,32.0,"McCormick Culinary Vanilla Extract, 32 fl oz -..."


In [18]:
X = test_data[['item_data', 'value']]

In [19]:
X['value'] = pd.to_numeric(X['value'], errors='coerce') 

/tmp/ipykernel_101/3801193570.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['value'] = pd.to_numeric(X['value'], errors='coerce')


In [20]:
X['value_standardized'] = scaler.fit_transform(X[['value']])

In [23]:
X.drop("value",axis=1,inplace=True)

In [24]:
X

,item_data,value_standardized
0,Rani 14-Spice Eshamaya's Mango Chutney (Indian...,-0.042133
1,Natural MILK TEA Flavoring extract by HALO PAN...,-0.049619
2,Honey Filled Hard Candy - Bulk Pack 2 Pounds -...,-0.023199
3,Vlasic Snack'mm's Kosher Dill 16 Oz (Pack of 2),-0.049619
4,"McCormick Culinary Vanilla Extract, 32 fl oz -...",-0.023199
...,...,...
74995,Good Seasons Zezty Italian Salad Dressing Mix ...,-0.049266
74996,"Colombina Swirled Love Tiger Pops, Strawberry ...",-0.045215
74997,"Kerns, Guava Nectar, 11.5 Fl Oz Can Kerns Guav...",-0.041252
74998,NY SPICE SHOP Licorice Candy - 1 Pound Red Lic...,-0.037290


In [26]:
X_test_emb = get_bert_embeddings(X['item_data'].tolist(), tokenizer, bert_model)

100%|██████████| 2344/2344 [04:40<00:00,  8.36it/s]


In [27]:
X_test_tensor = torch.tensor(X_test_emb, dtype=torch.float32)

In [29]:
with torch.no_grad():
    y_pred = t_model(X_test_tensor.to(device)).cpu().numpy()

In [30]:
pred_price = np.expm1(y_pred).clip(0)

In [36]:
pred_price

array([[18.361141 ],
       [13.664108 ],
       [15.591912 ],
       ...,
       [ 4.049253 ],
       [ 8.593312 ],
       [ 7.5369673]], dtype=float32)

In [39]:
gg=pred_price.flatten()
gg

array([18.361141 , 13.664108 , 15.591912 , ...,  4.049253 ,  8.593312 ,
        7.5369673], dtype=float32)

In [40]:
submission = pd.DataFrame({
    'sample_id': test_data['sample_id'],
    'price': gg
})

In [42]:
submission.shape

(75000, 2)

In [43]:
submission.to_csv('submission.csv', index=False)
print("Submission file saved as submission.csv")

Submission file saved as submission.csv
